In [1]:
import os
import re

from datasets import load_dataset, Dataset, load_from_disk
from BPETokenizer import BPETokenizer as tk

/opt/homebrew/anaconda3/envs/buildinglargelanguagemodel/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading the Pre-Train Dataset

In [2]:
dataset_dict = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", streaming=True)

In [3]:
dataset = dataset_dict['train'].take(10000)

In [4]:
sample_batch = dataset.take(10)

for _, row in enumerate(sample_batch):
    print(row['text'][:500])

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All species too; the lecture featured touching photos of a polar bear and a husky engaging playfully at a snowy outpost in northern Canada.) Stuart Brown, president of the National Institute for Play, was 

In [5]:
local_data_list = list(dataset_dict['train'].take(10000))

In [6]:
local_dataset = Dataset.from_list(local_data_list)

In [7]:
if not os.path.exists("local_fineweb_data"):
    local_dataset.save_to_disk("local_fineweb_data/hf_format")

In [8]:
local_dataset = load_from_disk("local_fineweb_data/hf_format")

In [9]:
for index in range(10):
    print(local_dataset[index]['text'][:500])

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All species too; the lecture featured touching photos of a polar bear and a husky engaging playfully at a snowy outpost in northern Canada.) Stuart Brown, president of the National Institute for Play, was 

In [10]:
total_tokens = sum(local_dataset['token_count'])
print(f"the training data has {total_tokens} tokens")

the training data has 10302766 tokens


**REGEX PATTERN And Tokenization By Row**

In [11]:
REGEX_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"""

### Building the Vocabulary

In [12]:
tokenizer = tk(REGEX_PATTERN, num_merges=1000)
tokenizer.build_vocab(local_dataset)
tokenized_with_ids = local_dataset.map(tokenizer.encode_row, num_proc=10)

Map (num_proc=10): 100%|██████████| 10000/10000 [01:35<00:00, 104.51 examples/s]


In [13]:
print(tokenizer.vocab_size)

2499


In [14]:
total = sum(len(row['token_ids']) for row in tokenized_with_ids)
print(total)


17933377


In [18]:
all_token_ids= []

for row in local_dataset:
    all_token_ids.extend(tokenizer.encode((row['text'])))


all_token_ids

[1693,
 2315,
 1783,
 1743,
 1539,
 1710,
 1524,
 72,
 2,
 41,
 1517,
 1738,
 1504,
 1546,
 1909,
 15,
 3,
 1631,
 1787,
 1530,
 2020,
 1635,
 1521,
 1525,
 1710,
 1524,
 72,
 1564,
 1705,
 1510,
 695,
 86,
 2495,
 86,
 15,
 1988,
 1740,
 1595,
 1544,
 1904,
 1827,
 1552,
 1523,
 1503,
 1520,
 1545,
 1530,
 1908,
 1783,
 1743,
 1774,
 17,
 2315,
 1783,
 1743,
 1774,
 1519,
 2450,
 87,
 1530,
 1504,
 1523,
 1503,
 1520,
 1545,
 1532,
 1660,
 82,
 1784,
 17,
 2,
 40,
 79,
 1741,
 1621,
 2079,
 695,
 86,
 2120,
 1617,
 1521,
 1519,
 1593,
 85,
 17,
 1574,
 1554,
 79,
 2123,
 1971,
 1506,
 1519,
 1529,
 1527,
 1639,
 1727,
 2298,
 1520,
 1591,
 1908,
 1783,
 1743,
 1774,
 1509,
 1584,
 71,
 1545,
 1666,
 1510,
 1525,
 1620,
 1536,
 2137,
 1519,
 1504,
 2391,
 17,
 1624,
 1506,
 2120,
 1617,
 1521,
 1519,
 1593,
 85,
 17,
 1647,
 1527,
 2046,
 2373,
 1965,
 1571,
 1609,
 1931,
 1629,
 1591,
 2466,
 2298,
 1520,
 1500,
 2287,
 1519,
 1908,
 1783,
 1743,
 1774,
 1572,
 1680,
 2106,
 2259,
 16

In [19]:
print(len(all_token_ids))

17933377


In [26]:
context_size = 4
enc_sample = all_token_ids[100:]

In [27]:
enc_sample

[1783,
 1743,
 1774,
 1509,
 1584,
 71,
 1545,
 1666,
 1510,
 1525,
 1620,
 1536,
 2137,
 1519,
 1504,
 2391,
 17,
 1624,
 1506,
 2120,
 1617,
 1521,
 1519,
 1593,
 85,
 17,
 1647,
 1527,
 2046,
 2373,
 1965,
 1571,
 1609,
 1931,
 1629,
 1591,
 2466,
 2298,
 1520,
 1500,
 2287,
 1519,
 1908,
 1783,
 1743,
 1774,
 1572,
 1680,
 2106,
 2259,
 1682,
 2419,
 1520,
 1530,
 1578,
 1669,
 81,
 1520,
 17,
 2,
 1693,
 1523,
 1503,
 1520,
 1545,
 2214,
 1622,
 75,
 1745,
 2050,
 1525,
 1523,
 1502,
 1904,
 2005,
 2152,
 1526,
 2259,
 1525,
 1534,
 2320,
 2174,
 76,
 1787,
 1519,
 1678,
 1583,
 92,
 1574,
 2207,
 1704,
 1530,
 1934,
 1526,
 2087,
 1523,
 2207,
 1899,
 1742,
 68,
 1654,
 1536,
 1548,
 1640,
 1805,
 1617,
 1777,
 2154,
 1570,
 1564,
 1705,
 1510,
 17,
 1733,
 2087,
 1546,
 1714,
 2495,
 2277,
 1994,
 1653,
 1913,
 76,
 1562,
 1552,
 2124,
 88,
 1583,
 1520,
 1532,
 2120,
 88,
 1555,
 1574,
 68,
 1694,
 1648,
 1644,
 1539,
 90,
 2454,
 1641,
 1678,
 1583,
 92,
 1657,
 1617,
 2034,
 

In [28]:
x = enc_sample [:context_size]
y = enc_sample [1: context_size  + 1]

In [29]:
x,y

([1783, 1743, 1774, 1509], [1743, 1774, 1509, 1584])

In [30]:
for index in range(1, context_size + 1):
    context = enc_sample[:index]
    desired = enc_sample[index]
    print(tokenizer.decode(context), "--------->", tokenizer.decode([desired]))

ep ---------> end
epend ---------> ence
ependence --------->  s
ependence s ---------> el
